# Chapitre 10 - Cours 3 : MapReduce, le shuffle et les jointures

MapReduce a rendu le traitement par lots distribué accessible : on écrit deux fonctions,
le framework se charge du reste (découpage, placement, tri distribué, relances).
C'est le modèle central de ce chapitre du livre, implémenté par Hadoop et repris par de nombreux systèmes.
Ce cours construit un MapReduce miniature (`B.run_mapreduce`) qui écrit de vrais
fichiers de shuffle, puis s'en sert pour les jointures et les regroupements. Il couvre
les sections *MapReduce Job Execution* (*Distributed execution of MapReduce*, *MapReduce workflows*), *Reduce-Side Joins and Grouping* (avec *Handling skew*) et *Map-Side Joins*.

**Plan.**

1. Le modèle MapReduce : un mappeur, un réducteur, et un tri entre les deux.
2. L'exécution distribuée : le shuffle, un tri distribué.
3. Enchaîner des jobs MapReduce.
4. Jointures et regroupements côté reduce (jointure tri-fusion).
5. L'asymétrie : clés chaudes et réducteurs à la traîne.
6. Jointures côté map : hachage diffusé, hachage partitionné, fusion.

In [1]:
import json
import os
import subprocess
import sys
import tempfile
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import batch as B
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, COLORS, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
lines = B.generate_access_log(200_000, n_urls=2000, s=1.1, seed=2)